In [ ]:
# Core
import numpy as np
import matplotlib.pyplot as plt

# Classical ML
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Metrics
from sklearn.metrics import accuracy_score, confusion_matrix

# Quantum
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.circuit.library import ZFeatureMap, TwoLocal
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit.algorithms.optimizers import COBYLA
from qiskit.primitives import Estimator
from qiskit_machine_learning.connectors import TorchConnector

In [63]:
IMAGE_SIZE = 200
BATCH_SIZE = 16

# Training transforms (with augmentation)
train_transform = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(200, scale=(0.95, 1.0)),
    transforms.ToTensor(),
])

# Test transforms (NO augmentation)
test_transform = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
])


train_data = datasets.ImageFolder(
    "../data/train", transform=train_transform
)
test_data = datasets.ImageFolder(
    "../data/test", transform=test_transform
)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

print("Classes:", train_data.classes)


Classes: ['covid', 'normal']


In [64]:
class ClassicalCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # (if you add more blocks later, no problem)
        )

        # Dynamically infer feature size
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 200, 200)
            out = self.features(dummy)
            feature_dim = out.view(1, -1).size(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 1)   # logits
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


In [65]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)
model = ClassicalCNN().to(device)

class_counts = np.bincount(train_data.targets)
neg, pos = class_counts  # class 0, class 1
pos_weight = torch.tensor(neg / pos, dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(
    optimizer, step_size=15, gamma=0.5
)

for epoch in range(40):
    model.train()
    total_loss = 0

    epoch_loss = 0
    num_samples = 0

    for x, y in train_loader:
        x, y = x.to(device), y.float().to(device)

        # LABEL SMOOTHING
        y = y * 0.9 + 0.05

        optimizer.zero_grad()
        preds = model(x).squeeze()
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * x.size(0)
        num_samples += x.size(0)

    print(f"Epoch {epoch+1}, Avg Loss: {epoch_loss / num_samples:.4f}")

    if epoch == 15:
        for param in model.features[0:4].parameters():
            param.requires_grad = False
        print("🔒 Frozen first conv block")

    scheduler.step()


Using device: mps
Epoch 1, Avg Loss: 0.7366
Epoch 2, Avg Loss: 0.5438
Epoch 3, Avg Loss: 0.5258
Epoch 4, Avg Loss: 0.5041
Epoch 5, Avg Loss: 0.4897
Epoch 6, Avg Loss: 0.4836
Epoch 7, Avg Loss: 0.4636
Epoch 8, Avg Loss: 0.4602
Epoch 9, Avg Loss: 0.4557
Epoch 10, Avg Loss: 0.4560
Epoch 11, Avg Loss: 0.4356
Epoch 12, Avg Loss: 0.4362
Epoch 13, Avg Loss: 0.4312
Epoch 14, Avg Loss: 0.4314
Epoch 15, Avg Loss: 0.4270
Epoch 16, Avg Loss: 0.4031
🔒 Frozen first conv block
Epoch 17, Avg Loss: 0.3899
Epoch 18, Avg Loss: 0.3865
Epoch 19, Avg Loss: 0.3847
Epoch 20, Avg Loss: 0.3803
Epoch 21, Avg Loss: 0.3745
Epoch 22, Avg Loss: 0.3715
Epoch 23, Avg Loss: 0.3638
Epoch 24, Avg Loss: 0.3620
Epoch 25, Avg Loss: 0.3563
Epoch 26, Avg Loss: 0.3493
Epoch 27, Avg Loss: 0.3618
Epoch 28, Avg Loss: 0.3587
Epoch 29, Avg Loss: 0.3494
Epoch 30, Avg Loss: 0.3451
Epoch 31, Avg Loss: 0.3423
Epoch 32, Avg Loss: 0.3341
Epoch 33, Avg Loss: 0.3239
Epoch 34, Avg Loss: 0.3314
Epoch 35, Avg Loss: 0.3325
Epoch 36, Avg Loss: 

In [66]:
# THRESHOLD SCAN (ANALYSIS)

from numpy import arange

model.eval()
probs_all, y_true = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x).squeeze()
        probs = torch.sigmoid(logits).cpu().numpy()
        probs_all.extend(probs.tolist())
        y_true.extend(y.numpy().tolist())

best_acc = 0
best_t = 0

for t in arange(0.35, 0.65, 0.01):
    preds = (np.array(probs_all) > t).astype(int)
    acc = accuracy_score(y_true, preds)
    if acc > best_acc:
        best_acc = acc
        best_t = t

print("Best threshold:", best_t)
print("Best CNN Accuracy:", best_acc)


Best threshold: 0.6000000000000002
Best CNN Accuracy: 0.705


In [67]:
# FINAL_THRESHOLD = 0.65
FINAL_THRESHOLD = best_t

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x).squeeze()
        preds = (torch.sigmoid(logits) > FINAL_THRESHOLD).int().cpu().numpy()

        y_pred.extend(preds.tolist())
        y_true.extend(y.numpy().tolist())

cnn_acc = accuracy_score(y_true, y_pred)
print("Classical CNN Accuracy:", cnn_acc)
print(confusion_matrix(y_true, y_pred))


Classical CNN Accuracy: 0.705
[[213  87]
 [ 90 210]]


**QCNN:**

In [68]:
class FeatureExtractor(nn.Module):
    def __init__(self, cnn_features):
        super().__init__()
        self.features = cnn_features

        with torch.no_grad():
            dummy = torch.zeros(1, 3, 200, 200).to(next(self.features.parameters()).device)
            out = self.features(dummy)
            feature_dim = out.view(1, -1).size(1)

        self.fc = nn.Linear(feature_dim, 4)  # quantum features

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


In [69]:
# class HybridQCNN(nn.Module):
#     def __init__(self, qnn):
#         super().__init__()
#         self.qnn = qnn
#         self.fc = nn.Linear(16, 1)  # 🔑 2 observables → 1 logit

#     def forward(self, x):
#         x = self.qnn(x)
#         return self.fc(x)
class HybridQCNN(nn.Module):
    def __init__(self, qnn):
        super().__init__()
        self.qnn = qnn
        self.fc1 = nn.Linear(4, 16)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        x = self.qnn(x)
        x = self.act(self.fc1(x))
        return self.fc2(x)



In [70]:
feature_model = FeatureExtractor(model).to(device)
feature_model.eval()

X_train, y_train = [], []

with torch.no_grad():
    for x, y in train_loader:
        x = x.to(device)
        features = feature_model(x).cpu().numpy()
        X_train.extend(features)
        y_train.extend(y.numpy())


In [71]:
# TRAIN FEATURE NORMALIZATION (PER-FEATURE)
X_train = np.array(X_train)
y_train = np.array(y_train)

# compute per-feature min/max on TRAIN ONLY
mins = X_train.min(axis=0)
maxs = X_train.max(axis=0)

# normalize each feature independently
X_train = (X_train - mins) / (maxs - mins + 1e-8)

# map to quantum angle range
X_train = X_train * np.pi   # [0, pi]

# BALANCE DATA FOR QCNN
from sklearn.utils import resample

# separate classes
pos = X_train[y_train == 1]
neg = X_train[y_train == 0]

# take equal samples
n = min(len(pos), len(neg))

X_bal = np.vstack([pos[:n], neg[:n]])
y_bal = np.hstack([np.ones(n), np.zeros(n)])

# shuffle
idx = np.random.permutation(len(X_bal))
X_bal = X_bal[idx]
y_bal = y_bal[idx]

# limit QCNN samples
X_train_q = X_bal[:200]
y_train_q = y_bal[:200]

print("QCNN class balance:", np.mean(y_train_q))  # should be ~0.5


QCNN class balance: 0.565


In [72]:
num_qubits = 4

feature_map = ZFeatureMap(num_qubits)
ansatz = TwoLocal(
    num_qubits,
    rotation_blocks=["ry", "rz"],
    entanglement_blocks="cx",
    reps=2,     # increase depth
    entanglement="full" # full or linear
)

qc = QuantumCircuit(num_qubits)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)
# qc.draw("mpl")

In [73]:
# backend = AerSimulator()
# optimizer = COBYLA(maxiter=100)
# estimator = Estimator()
from qiskit.primitives import Estimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_aer.primitives import Estimator as AerEstimator

# estimator = AerEstimator(run_options={"shots": 1024})
from qiskit.primitives import Estimator
estimator = Estimator()   # analytic, NO shots

observables = [
    SparsePauliOp.from_list([("ZIII", 1.0)]),
    SparsePauliOp.from_list([("IZII", 1.0)]),
    SparsePauliOp.from_list([("IIZI", 1.0)]),
    SparsePauliOp.from_list([("IIIZ", 1.0)]),
]

qnn = EstimatorQNN(
    circuit=qc,
    observables=observables,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
    estimator=estimator,
)

base_qnn = TorchConnector(qnn).to(device)
model_qcnn = HybridQCNN(base_qnn).to(device)

# vqc.fit(X_train, y_train)


In [74]:
optimizer_q = optim.Adam(model_qcnn.parameters(), lr=0.005)
criterion_q = nn.BCEWithLogitsLoss()
X_q = torch.tensor(X_train_q, dtype=torch.float32).to(device)
y_q = torch.tensor(y_train_q, dtype=torch.float32).view(-1, 1).to(device)

batch_size = 16

for epoch in range(30):
    perm = torch.randperm(len(X_q))
    total_loss = 0

    for i in range(0, len(X_q), batch_size):
        idx = perm[i:i+batch_size]
        xb = X_q[idx]
        yb = y_q[idx]

        optimizer_q.zero_grad()
        outputs = model_qcnn(xb)
        loss = criterion_q(outputs, yb)
        loss.backward()
        optimizer_q.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Epoch 0, Loss: 8.4220
Epoch 1, Loss: 7.5722
Epoch 2, Loss: 6.6682
Epoch 3, Loss: 5.8431
Epoch 4, Loss: 5.1377
Epoch 5, Loss: 4.6308
Epoch 6, Loss: 4.5361
Epoch 7, Loss: 3.9114
Epoch 8, Loss: 3.6133
Epoch 9, Loss: 3.4415
Epoch 10, Loss: 3.2794
Epoch 11, Loss: 3.1284
Epoch 12, Loss: 3.1017
Epoch 13, Loss: 3.0700
Epoch 14, Loss: 2.8808
Epoch 15, Loss: 2.9091
Epoch 16, Loss: 2.8456
Epoch 17, Loss: 2.8021
Epoch 18, Loss: 2.6862
Epoch 19, Loss: 2.9158
Epoch 20, Loss: 2.6155
Epoch 21, Loss: 2.6516
Epoch 22, Loss: 2.5922
Epoch 23, Loss: 2.5573
Epoch 24, Loss: 2.6302
Epoch 25, Loss: 2.6919
Epoch 26, Loss: 2.6396
Epoch 27, Loss: 2.5290
Epoch 28, Loss: 2.4917
Epoch 29, Loss: 2.7002


In [75]:
X_test, y_test = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        feats = feature_model(x).cpu().numpy()
        X_test.extend(feats)
        y_test.extend(y.numpy())

X_test = np.array(X_test)
y_test = np.array(y_test)

# NORMALIZE TEST USING TRAIN STATS
X_test = (X_test - mins) / (maxs - mins + 1e-8)
X_test = X_test * np.pi


In [ ]:
# Convert to torch
X_t = torch.tensor(X_test, dtype=torch.float32).to(device)

model_qcnn.eval()

with torch.no_grad():
    outputs = model_qcnn(X_t)
    probs = torch.sigmoid(outputs / 1.5).cpu().numpy()
    preds = (probs > 0.5).astype(int)

acc = accuracy_score(y_test, preds)
print("QCNN Accuracy:", acc)

cnn_probs = np.array(probs_all)

qcnn_probs = probs.flatten()

best_acc = 0
best_alpha = 0
best_t = 0

for alpha in np.arange(0.6, 0.9, 0.05):   # CNN weight
    for t in np.arange(0.5, 0.7, 0.02):

        final_probs = alpha * cnn_probs + (1 - alpha) * qcnn_probs
        final_preds = (final_probs > t).astype(int)

        acc = accuracy_score(y_true, final_preds)

        if acc > best_acc:
            best_acc = acc
            best_alpha = alpha
            best_t = t

print("BEST HYBRID ACC:", best_acc)
print("Best alpha:", best_alpha)
print("Best threshold:", best_t)

QCNN Accuracy: 0.695
🔥 BEST HYBRID ACC: 0.7066666666666667
Best alpha: 0.65
Best threshold: 0.6200000000000001
